# ML BTC

## Imports

In [11]:
import os
import requests
from datetime import datetime

import kagglehub
import polars as pl

## Dataset loading

### Bitcoin historical data

In [12]:
path = kagglehub.dataset_download("adilshamim8/bitcoin-historical-data")
filename = "Bitcoin_history_data.csv"
filepath = os.path.join(path, filename)
df_btc = pl.read_csv(filepath)
df_btc

Date,Close,High,Low,Open,Volume
str,f64,f64,f64,f64,i64
"""2014-09-17""",457.334015,468.174011,452.421997,465.864014,21056800
"""2014-09-18""",424.440002,456.859985,413.104004,456.859985,34483200
"""2014-09-19""",394.79599,427.834991,384.532013,424.102997,37919700
"""2014-09-20""",408.903992,423.29599,389.882996,394.673004,36863600
"""2014-09-21""",398.821014,412.425995,393.181,408.084991,26580100
…,…,…,…,…,…
"""2026-02-16""",68843.15625,70067.234375,67301.585938,68782.398438,33618145426
"""2026-02-17""",67494.21875,69201.867188,66615.28125,68843.09375,34866936040
"""2026-02-18""",66425.320312,68434.429688,65845.898438,67488.023438,33094301643


### Fear and greed index

In [13]:
url = "https://api.alternative.me/fng/?limit=0&format=json"
response = requests.get(url)
data = response.json()["data"]

rows = []
for row in data:
    timestamp = int(row["timestamp"])
    date = str(datetime.fromtimestamp(timestamp).date())
    value = row["value_classification"]
    rows.append({"Date": date, "Fear and greed": value})

df_fear_and_greed = pl.DataFrame(rows)
df_fear_and_greed

Date,Fear and greed
str,str
"""2026-03-03""","""Extreme Fear"""
"""2026-03-02""","""Extreme Fear"""
"""2026-03-01""","""Extreme Fear"""
"""2026-02-28""","""Extreme Fear"""
"""2026-02-27""","""Extreme Fear"""
…,…
"""2018-02-05""","""Extreme Fear"""
"""2018-02-04""","""Extreme Fear"""
"""2018-02-03""","""Fear"""


### Dataset merging

In [14]:
df = df_btc.join(df_fear_and_greed, on="Date", how="outer")
df

/tmp/ipykernel_19071/317962971.py:1: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  df = df_btc.join(df_fear_and_greed, on="Date", how="outer")


Date,Close,High,Low,Open,Volume,Date_right,Fear and greed
str,f64,f64,f64,f64,i64,str,str
"""2014-09-17""",457.334015,468.174011,452.421997,465.864014,21056800,null,null
"""2014-09-18""",424.440002,456.859985,413.104004,456.859985,34483200,null,null
"""2014-09-19""",394.79599,427.834991,384.532013,424.102997,37919700,null,null
"""2014-09-20""",408.903992,423.29599,389.882996,394.673004,36863600,null,null
"""2014-09-21""",398.821014,412.425995,393.181,408.084991,26580100,null,null
…,…,…,…,…,…,…,…
null,null,null,null,null,null,"""2026-02-22""","""Extreme Fear"""
null,null,null,null,null,null,"""2026-03-02""","""Extreme Fear"""
null,null,null,null,null,null,"""2026-02-28""","""Extreme Fear"""


## Labeling

The current dataset does not contain any labels, so we need to assign the following labels to each row:
- Buy zone: favorable conditions for buying
- Sell zone: favorable conditions for selling
- Neutral zone: hold (HODL)

Methodology :
1. The dataset is divided into distinct periods.
2. Each period ends when a 50% drawdown from the ATH (All-Time High), calculated using the closing price, occurs.
3. A period is defined as follows:
    - From the beginning of the dataset to the first 50% drawdown
    - From one 50% drawdown to the next 50% drawdown
    - From the last 50% drawdown to the end of the dataset
4. For each period:
    1. The lowest and highest prices are identified.
    2. A Fibonacci retracement is applied from the highest price to the lowest price.
    3. The zones are defined as follows:
        - 0 ≤ price ≤ 0.236: Buy zone
        - 0.236 < price < 0.764: Neutral zone
        - 0.764 ≤ price ≤ 1: Sell zone

In [15]:
df_tmp = df.clone()
df_tmp = df_tmp.drop_nulls(subset=["Date", "Close"])
df_tmp = df_tmp.select(["Date", "Close"])
df_tmp = df_tmp.with_columns(
    pl.col("Date").str.strptime(pl.Datetime, format="%Y-%m-%d")
)
df_tmp = df_tmp.sort("Date")

# calculate ATH
df_tmp = df_tmp.with_columns(
    pl.col("Close").cum_max()
    .alias("ATH")
)

# calculate drawdown from ATH (0 to -1)
df_tmp = df_tmp.with_columns(
    (pl.col("Close") / pl.col("ATH") - 1).alias("Drawdown")
)

# determine periods based on drawdown >50% after a new ATH ()
df_tmp_pd = df_tmp.to_pandas()

period_id = 0
last_ath_for_period = df_tmp_pd.loc[0, "ATH"]
period_ids = []

for idx, row in df_tmp_pd.iterrows():
    if row["Drawdown"] < -0.5 and row["ATH"] > last_ath_for_period:
        period_id += 1
        last_ath_for_period = row["ATH"]
    period_ids.append(period_id)

df_tmp_pd["Period ID"] = period_ids
df_tmp = pl.from_pandas(df_tmp_pd)

# calculate the local low for each period
df_tmp = df_tmp.with_columns(
    pl.col("Close").min().over("Period ID").alias("Local Low")
)

# calculate Fibonacci level
df_tmp = df_tmp.with_columns(
    (
        (pl.col("Close") - pl.col("Local Low")) / (pl.col("ATH") - pl.col("Local Low"))
    )
    .alias("Fibbonacci level")
)

# label based on Fibonacci retracement
df_tmp = df_tmp.with_columns(
    pl.when(pl.col("Fibbonacci level") <= 0.236).then(pl.lit("Buy zone"))
        .when(pl.col("Fibbonacci level") >= 0.764).then(pl.lit("Sell zone"))
        .otherwise(pl.lit("Neutral Zone"))
    .alias("Label")
)

periods = df_tmp.group_by("Period ID").agg([
          pl.col("Date").min().alias("Start date"),
          pl.col("Date").max().alias("End date"),
          pl.col("ATH").max(),
          pl.col("Local Low").min()
    ]).sort("Period ID")
print(periods)

# merging
df_tmp = df_tmp.select(["Date", "Label"])
df_tmp = df_tmp.with_columns(
    pl.col("Date").dt.strftime("%Y-%m-%d")
    .alias("Date")
)
df = df.join(df_tmp, on="Date", how="inner")
df


shape: (4, 5)
┌───────────┬─────────────────────┬─────────────────────┬──────────────┬──────────────┐
│ Period ID ┆ Start date          ┆ End date            ┆ ATH          ┆ Local Low    │
│ ---       ┆ ---                 ┆ ---                 ┆ ---          ┆ ---          │
│ i64       ┆ datetime[μs]        ┆ datetime[μs]        ┆ f64          ┆ f64          │
╞═══════════╪═════════════════════╪═════════════════════╪══════════════╪══════════════╡
│ 0         ┆ 2014-09-17 00:00:00 ┆ 2018-01-31 00:00:00 ┆ 19497.400391 ┆ 178.102997   │
│ 1         ┆ 2018-02-01 00:00:00 ┆ 2021-06-20 00:00:00 ┆ 63503.457031 ┆ 3236.761719  │
│ 2         ┆ 2021-06-21 00:00:00 ┆ 2022-05-08 00:00:00 ┆ 67566.828125 ┆ 29807.347656 │
│ 3         ┆ 2022-05-09 00:00:00 ┆ 2026-02-20 00:00:00 ┆ 124752.53125 ┆ 15787.28418  │
└───────────┴─────────────────────┴─────────────────────┴──────────────┴──────────────┘


Date,Close,High,Low,Open,Volume,Date_right,Fear and greed,Label
str,f64,f64,f64,f64,i64,str,str,str
"""2014-09-17""",457.334015,468.174011,452.421997,465.864014,21056800,null,null,"""Sell zone"""
"""2014-09-18""",424.440002,456.859985,413.104004,456.859985,34483200,null,null,"""Sell zone"""
"""2014-09-19""",394.79599,427.834991,384.532013,424.102997,37919700,null,null,"""Sell zone"""
"""2014-09-20""",408.903992,423.29599,389.882996,394.673004,36863600,null,null,"""Sell zone"""
"""2014-09-21""",398.821014,412.425995,393.181,408.084991,26580100,null,null,"""Sell zone"""
…,…,…,…,…,…,…,…,…
"""2026-02-16""",68843.15625,70067.234375,67301.585938,68782.398438,33618145426,"""2026-02-16""","""Extreme Fear""","""Neutral Zone"""
"""2026-02-17""",67494.21875,69201.867188,66615.28125,68843.09375,34866936040,"""2026-02-17""","""Extreme Fear""","""Neutral Zone"""
"""2026-02-18""",66425.320312,68434.429688,65845.898438,67488.023438,33094301643,"""2026-02-18""","""Extreme Fear""","""Neutral Zone"""


## Technical indicators calculation

These indicators are calculated before the train/test split because they are based only on the preceding N days of data.
Without this historical data, it would not be possible to compute these indicators.

### Weekly data

Some indicators are calculated on a weekly basis because this is more appropriate for long-term analysis.

In [16]:
df_tmp = df.clone()

df_tmp = df_tmp.with_columns(
    pl.col("Date").str.strptime(pl.Datetime, format="%Y-%m-%d")
)

df_tmp = df_tmp.with_columns(
    pl.col("Date").dt.strftime("%Y-%U")
    .alias("Week")
)

# sunday = 7
df_tmp = df_tmp.filter(pl.col("Date").dt.weekday() == 7) 
df_weekly = df_tmp.select(["Week", "Close"])
df_weekly

Week,Close
str,f64
"""2014-38""",398.821014
"""2014-39""",377.181
"""2014-40""",320.51001
"""2014-41""",378.549011
"""2014-42""",389.54599
…,…
"""2026-03""",93634.429688
"""2026-04""",86572.21875
"""2026-05""",76974.445312


### Relative Strength Index (RSI)

This index measures both the strength and the speed of a price movement.
A value above 70 indicates that BTC is overbought, while a value below 30 indicates that BTC is oversold.
For the long term, the weekly RSI is the most appropriate indicator.